In [9]:
from src.biotools.fasta_tools import *
seq=ExtractSequence('P04529','../uvsx/data/curated_database/filtered_curated_database.fasta')

In [10]:
walker_a=seq[58:67]
walker_b=seq[137:143]

In [18]:
print(f'Walker A motif: {walker_a}')
print(f'Walker B motif: {walker_b}')


Walker A motif: AGPSKSFKS
Walker B motif: VVVFID


In [12]:
import re
#looking for walker A in the gold standards

files = [
    '../uvsx/data/references/P04529.fasta',
    '../uvsx/data/references/9GBG.fasta',
    '../uvsx/data/references/7Z3M.fasta'
]

results = {}

for file in files:
    seq = read_fasta(file)
    name = file.split('/')[-1].split('.')[0]

    matches = []
    for match in re.finditer(r"[AVILMFWY]{4}DE?", seq):
        matches.append({
            "match": match.group(),
            "start": match.start(),
            "end": match.end()
        })

    results[name] = matches

print(results)


{'P04529': [{'match': 'VVFID', 'start': 138, 'end': 143}], '9GBG': [], '7Z3M': [{'match': 'LFILD', 'start': 156, 'end': 161}]}


In [45]:
import pandas as pd
df=pd.read_csv('../uvsx/data/curated_database/cleaned_metadata.csv')


In [46]:
#Finding Walker A motif in whole database
def MotifDatabaseSearch(accessions, database, regex):
    results={}
    present=[]
    for acc in accessions:
        seq = ExtractSequence(acc, database)

        matches = []
        for match in re.finditer(regex, seq):
            matches.append({
                "match": match.group(),
                "start": match.start(),
                "end": match.end()
            })

        results[acc] = matches
        if len(matches)==0:
            present.append(0)
        else:
            present.append(1)

    return results, present



In [54]:
a_results, a_present=MotifDatabaseSearch(df['accession'], '../uvsx/data/curated_database/cleaned_curated_database.fasta', r"[GA]....[GF]K[TS]")

In [55]:
b_results, b_present=MotifDatabaseSearch(df['accession'], '../uvsx/data/curated_database/cleaned_curated_database.fasta', r"[AVILMFWY]{4}DE?")

In [58]:
sum(b_present)

13007

In [43]:
a_absent=[]

for acc in df['accession']:
    if len(a_results[acc]) == 0:
        a_absent.append(acc)
print(f'Number of Sequences without Walker A: {len(a_absent)}')

a_fs=[]
for acc in df['accession']:
    for match in a_results[acc]:
        seq=match.get('match')
        if seq[-3] == 'F':
            a_fs.append(acc)

print(f'Number of Sequences With F instead of G: {len(a_fs)}')

Number of Sequences without Walker A: 20549
Number of Sequences With F instead of G: 0


In [16]:
# Finding walker B motif
b_results={}
b_present=[]
for acc in df['accession']:
    seq = ExtractSequence(acc, '../uvsx/data/curated_database/cleaned_curated_database.fasta')

    matches = []
    for match in re.finditer(r"[AVILMFWY]{4}DE?", seq):
        matches.append({
            "match": match.group(),
            "start": match.start(),
            "end": match.end()
        })

    b_results[acc] = matches
    if len(matches)==0:
        print(f'{acc} WALKER B NOT FOUND')
        b_present.append(0)
    else:
        b_present.append(1)

P32270 WALKER B NOT FOUND
YDI96965 WALKER B NOT FOUND
YDI96730 WALKER B NOT FOUND
YDF88887 WALKER B NOT FOUND
YDF81549 WALKER B NOT FOUND
YDF75915 WALKER B NOT FOUND
YDF71083 WALKER B NOT FOUND
YDF68759 WALKER B NOT FOUND
YDF68054 WALKER B NOT FOUND
YDF61728 WALKER B NOT FOUND
YCZ41734 WALKER B NOT FOUND
YCQ84975 WALKER B NOT FOUND
O21960 WALKER B NOT FOUND
O21959 WALKER B NOT FOUND
O21958 WALKER B NOT FOUND
O21957 WALKER B NOT FOUND
O21956 WALKER B NOT FOUND
O21955 WALKER B NOT FOUND
O21954 WALKER B NOT FOUND
O21953 WALKER B NOT FOUND
O21952 WALKER B NOT FOUND
O21951 WALKER B NOT FOUND
O21950 WALKER B NOT FOUND
O21949 WALKER B NOT FOUND
O21947 WALKER B NOT FOUND
O21948 WALKER B NOT FOUND
Q06728 WALKER B NOT FOUND
Q06727 WALKER B NOT FOUND
P04537 WALKER B NOT FOUND
P41662 WALKER B NOT FOUND
YCJ24104 WALKER B NOT FOUND
XSD99949 WALKER B NOT FOUND
UNA07219 WALKER B NOT FOUND
UMO75314 WALKER B NOT FOUND
YBX93090 WALKER B NOT FOUND
YBX92995 WALKER B NOT FOUND
YBX92994 WALKER B NOT FOUND
YB

In [19]:
absent=[]
for acc in df['accession']:
    if len(results[acc]) == 0:
        absent.append(acc)
print(f'Number of Sequences without Walker B: {len(absent)}')



KeyError: 'P32270'

In [20]:
import pandas as pd
hits=pd.read_csv('../uvsx/data/hmms/results/accession_hits.csv')

In [23]:
hits

,accession,full_sequence_hmm_results,PF00154_custom_hmm_results,PF00154_hmm_results,PF21134_custom_hmm_results,PF21134_hmm_results,walker_a,walker_b
0,P32270,0,0,0,0,0,1,0
1,P20315,0,0,0,0,0,0,1
2,UYE93698,0,0,0,0,0,0,1
3,YDI96965,0,0,1,0,0,0,0
4,YDI96730,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...
20544,Q37876,0,0,0,0,0,0,0
20545,Q37877,0,0,0,0,0,0,1
20546,C9DGL1,0,0,0,0,0,0,1
20547,Q9T0S5,0,0,0,0,0,0,0


In [61]:
def WriteScoringtoCSV(score_csv, score_vector, score_name):
    df = pd.read_csv(score_csv)
    df[score_name]=score_vector
    df.to_csv(score_csv, index=False)

In [ ]:
WriteScoringtoCSV('../')

In [22]:
top_hits=[]
for index, row in hits.iterrows():
    if sum(row.drop('accession')) == 7:
        top_hits.append(row['accession'])

In [24]:
hits[hits['accession']=='YDI88709']

,accession,full_sequence_hmm_results,PF00154_custom_hmm_results,PF00154_hmm_results,PF21134_custom_hmm_results,PF21134_hmm_results,walker_a,walker_b
8,YDI88709,1,1,1,1,1,1,1


In [25]:
ExtractSequence('YDI88709', '../uvsx/data/curated_database/cleaned_curated_database.fasta')

'MSDLKSRLIKASTSKMTADLTKSKLFNNRDEVPTRIPMLNIALGGALNAGLQSGLTIFAAPSKHFKTLFGLTMVAAYMKKYKDAICLFYDSEFGASESYFRSMGVDLDRVVHTPIQSVEQLKVDMTNQLDAIERGDKVIIFIDSIGNTASKKETEDALNEKVVGDMSRAKALKSLFRIVTPYLTIKDIPCVAINHTAMEIGGLYPKEIMGGGTGILYSANTVFFISKRQVKEGTELTGYDFTLKAEKSRTVKEKSTFPITVNFDGGIDPFSGLLEMATEIGFVVKPKAGWYAREFLDEETGEMIREEKSWRAKATDCVEFWGPLFKHKPFRDAIETKYKLGAISSIKEVDDAVNDLINCKATTKVPVKTSDAPSAADIENDLDEMEDFDE'

In [26]:
ExtractSequence('P04529', '../uvsx/data/curated_database/cleaned_curated_database.fasta')

'MSDLKSRLIKASTSKLTAELTASKFFNEKDVVRTKIPMMNIALSGEITGGMQSGLLILAGPSKSFKSNFGLTMVSSYMRQYPDAVCLFYDSEFGITPAYLRSMGVDPERVIHTPVQSLEQLRIDMVNQLDAIERGEKVVVFIDSLGNLASKKETEDALNEKVVSDMTRAKTMKSLFRIVTPYFSTKNIPCIAINHTYETQEMFSKTVMGGGTGPMYSADTVFIIGKRQIKDGSDLQGYQFVLNVEKSRTVKEKSKFFIDVKFDGGIDPYSGLLDMALELGFVVKPKNGWYAREFLDEETGEMIREEKSWRAKDTNCTTFWGPLFKHQPFRDAIKRAYQLGAIDSNEIVEAEVDELINSKVEKFKSPESKSKSAADLETDLEQLSDMEEFNE'

In [27]:
len(top_hits)

626

In [28]:
for acc in top_hits:
    seq=ExtractSequence(acc, '../uvsx/data/curated_database/cleaned_curated_database.fasta')
    if len(seq) < 300 and len(seq) > 400:
        print(f'{acc} has abbererant sequence\n{seq}')


Okay so 626 sequences have a hit for all 7 checks. However with walker a and b these hits may be in sections of the protein that dont make sense
Potentially need to do filtering for motif hits that are in sketchy locations

In [33]:
#Rational design paper has loops 1 and 2
seq=ExtractSequence('P04529', '../uvsx/data/curated_database/cleaned_curated_database.fasta')
ploop=seq[58:67]
loop1=seq[151:165]
loop2=seq[197:211]
print(f'Loop 1: {loop1}')
print(f'Loop 2: {loop2}')

Loop 1: KETEDALNEKVVSD
Loop 2: ETQEMFSKTVMGGG
